# 📊 01 — Exploratory Data Analysis (EDA)

**Proyek:** Segmentasi UMKM Kota Bandung menggunakan K-Means Clustering

**Tujuan Notebook Ini:**
- Memahami struktur dan karakteristik dataset mentah
- Mengidentifikasi missing value, duplikasi, dan anomali
- Menampilkan distribusi variabel kunci (rating, jumlah ulasan, kategori)
- Visualisasi data untuk memahami pola awal

> **Catatan:** Notebook ini HANYA berisi eksplorasi dan visualisasi. Proses cleaning dilakukan di notebook `02_Preprocessing.ipynb`.

## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print('Library berhasil dimuat.')

## 2. Memuat Dataset Mentah

Dataset diperoleh dari scraping Google Maps via **Apify Google Places Scraper** oleh 3 anggota:
- Indra (2 run)
- Dwi (3 run)
- Rajif (4 run)

Setiap run menghasilkan 2 file:
1. **Info Tempat** (overview) — data listing UMKM
2. **Teks Ulasan** (review) — ulasan pelanggan per listing

In [ ]:
RAW_PATH = os.path.join('..', 'data', 'raw')

# Daftar file mentah
raw_files = [f for f in os.listdir(RAW_PATH) if f.endswith('.csv')]
print(f'Jumlah file CSV mentah: {len(raw_files)}')
print()
for f in sorted(raw_files):
    size_mb = os.path.getsize(os.path.join(RAW_PATH, f)) / (1024 * 1024)
    print(f'  {f} ({size_mb:.2f} MB)')

### 2.1 Memuat Dataset Gabungan Utama

File `data_umkm_bandung.csv` adalah dataset gabungan utama yang berisi data info tempat dari semua anggota.

In [ ]:
df_main = pd.read_csv(os.path.join(RAW_PATH, 'data_umkm_bandung.csv'))
print(f'Shape dataset utama: {df_main.shape}')
print(f'Jumlah baris: {df_main.shape[0]:,}')
print(f'Jumlah kolom: {df_main.shape[1]}')

## 3. Struktur Data

In [ ]:
print('=== INFO DATASET ===')
df_main.info()

In [ ]:
print('=== 5 DATA PERTAMA ===')
df_main.head()

In [ ]:
print('=== TIPE DATA SETIAP KOLOM ===')
df_main.dtypes

In [ ]:
print('=== STATISTIK DESKRIPTIF (NUMERIK) ===')
df_main.describe()

In [ ]:
print('=== STATISTIK DESKRIPTIF (KATEGORIKAL) ===')
df_main.describe(include='object')

## 4. Analisis Missing Value

In [ ]:
missing = df_main.isnull().sum()
missing_pct = (missing / len(df_main) * 100).round(2)
missing_df = pd.DataFrame({
    'Jumlah Missing': missing,
    'Persentase (%)': missing_pct
}).sort_values('Jumlah Missing', ascending=False)

print('=== MISSING VALUE PER KOLOM ===')
missing_df[missing_df['Jumlah Missing'] > 0]

In [ ]:
# Visualisasi missing value
cols_with_missing = missing_df[missing_df['Jumlah Missing'] > 0].index.tolist()
if cols_with_missing:
    fig, ax = plt.subplots(figsize=(12, max(4, len(cols_with_missing) * 0.4)))
    missing_df.loc[cols_with_missing, 'Persentase (%)'].plot(
        kind='barh', color='coral', edgecolor='black', ax=ax
    )
    ax.set_xlabel('Persentase Missing (%)')
    ax.set_title('Persentase Missing Value per Kolom')
    plt.tight_layout()
    plt.show()
else:
    print('Tidak ada missing value.')

## 5. Analisis Duplikasi

In [ ]:
n_dup = df_main.duplicated().sum()
print(f'Jumlah baris duplikat penuh: {n_dup:,}')

if 'title' in df_main.columns:
    n_titles = df_main['title'].nunique()
    print(f'Jumlah judul unik: {n_titles:,}')
    print(f'Kemungkinan duplikat berdasarkan judul: {len(df_main) - n_titles:,}')

## 6. Distribusi Rating (totalScore)

In [ ]:
if 'totalScore' in df_main.columns:
    score_col = pd.to_numeric(df_main['totalScore'], errors='coerce')
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram
    axes[0].hist(score_col.dropna(), bins=20, color='steelblue', edgecolor='black', alpha=0.8)
    axes[0].set_xlabel('Rating (totalScore)')
    axes[0].set_ylabel('Frekuensi')
    axes[0].set_title('Distribusi Rating UMKM')
    axes[0].axvline(score_col.mean(), color='red', linestyle='--', label=f'Mean: {score_col.mean():.2f}')
    axes[0].legend()
    
    # Boxplot
    axes[1].boxplot(score_col.dropna(), vert=True)
    axes[1].set_ylabel('Rating')
    axes[1].set_title('Boxplot Rating UMKM')
    
    plt.tight_layout()
    plt.show()
    
    print(f'\nStatistik Rating:')
    print(f'  Mean   : {score_col.mean():.2f}')
    print(f'  Median : {score_col.median():.2f}')
    print(f'  Min    : {score_col.min():.2f}')
    print(f'  Max    : {score_col.max():.2f}')
    print(f'  Std    : {score_col.std():.2f}')

## 7. Distribusi Jumlah Ulasan (reviewsCount)

In [ ]:
if 'reviewsCount' in df_main.columns:
    review_col = pd.to_numeric(df_main['reviewsCount'], errors='coerce')
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram (log scale)
    axes[0].hist(np.log1p(review_col.dropna()), bins=30, color='seagreen', edgecolor='black', alpha=0.8)
    axes[0].set_xlabel('log(1 + reviewsCount)')
    axes[0].set_ylabel('Frekuensi')
    axes[0].set_title('Distribusi Jumlah Ulasan (Log Scale)')
    
    # Boxplot
    axes[1].boxplot(review_col.dropna(), vert=True)
    axes[1].set_ylabel('Jumlah Ulasan')
    axes[1].set_title('Boxplot Jumlah Ulasan')
    
    plt.tight_layout()
    plt.show()
    
    print(f'\nStatistik Jumlah Ulasan:')
    print(f'  Mean   : {review_col.mean():.0f}')
    print(f'  Median : {review_col.median():.0f}')
    print(f'  Min    : {review_col.min():.0f}')
    print(f'  Max    : {review_col.max():.0f}')

## 8. Distribusi Kategori Usaha

In [ ]:
if 'categoryName' in df_main.columns:
    cat_counts = df_main['categoryName'].value_counts().head(20)
    
    fig, ax = plt.subplots(figsize=(12, 8))
    cat_counts.plot(kind='barh', color='mediumpurple', edgecolor='black', ax=ax)
    ax.set_xlabel('Jumlah UMKM')
    ax.set_ylabel('Kategori Usaha')
    ax.set_title('Top 20 Kategori Usaha UMKM Kota Bandung')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
    
    print(f'\nTotal kategori unik: {df_main["categoryName"].nunique()}')

## 9. Distribusi Wilayah (City)

In [ ]:
if 'city' in df_main.columns:
    city_counts = df_main['city'].value_counts().head(15)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    city_counts.plot(kind='bar', color='darkorange', edgecolor='black', ax=ax)
    ax.set_xlabel('Kota')
    ax.set_ylabel('Jumlah Listing')
    ax.set_title('Distribusi Listing per Kota')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 10. Korelasi Antar Variabel Numerik

In [ ]:
numeric_cols = df_main.select_dtypes(include=[np.number]).columns.tolist()
if len(numeric_cols) >= 2:
    corr = df_main[numeric_cols].corr()
    
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f', ax=ax)
    ax.set_title('Heatmap Korelasi Variabel Numerik')
    plt.tight_layout()
    plt.show()

## 11. Scatter Plot Rating vs Jumlah Ulasan

In [ ]:
if 'totalScore' in df_main.columns and 'reviewsCount' in df_main.columns:
    score_col = pd.to_numeric(df_main['totalScore'], errors='coerce')
    review_col = pd.to_numeric(df_main['reviewsCount'], errors='coerce')
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.scatter(score_col, np.log1p(review_col), alpha=0.4, s=20, color='teal')
    ax.set_xlabel('Rating (totalScore)')
    ax.set_ylabel('log(1 + reviewsCount)')
    ax.set_title('Scatter Plot: Rating vs Jumlah Ulasan')
    plt.tight_layout()
    plt.show()

## 12. Contoh Data Review (Ulasan)

Melihat beberapa contoh data ulasan mentah untuk memahami struktur teks yang akan dianalisis NLP.

In [ ]:
# Memuat contoh file review
review_files = [f for f in raw_files if 'review' in f.lower() or 'ada_text' in f.lower()]
if review_files:
    df_review_sample = pd.read_csv(os.path.join(RAW_PATH, review_files[0]))
    print(f'Contoh file review: {review_files[0]}')
    print(f'Shape: {df_review_sample.shape}')
    print(f'\nKolom: {list(df_review_sample.columns[:15])}')
    if 'text' in df_review_sample.columns:
        print(f'\n--- Contoh 5 Ulasan Pertama ---')
        for i, txt in enumerate(df_review_sample['text'].dropna().head(5)):
            print(f'\n[{i+1}] {str(txt)[:200]}...' if len(str(txt)) > 200 else f'\n[{i+1}] {txt}')
else:
    print('File review tidak ditemukan untuk preview.')

## 13. Kesimpulan EDA

Berdasarkan eksplorasi data di atas:

1. **Dataset** terdiri dari data listing UMKM hasil scraping Google Maps via Apify dari 3 anggota (9 run total).
2. **Variabel kunci** untuk modeling: `totalScore` (rating), `reviewsCount` (jumlah ulasan), dan teks ulasan (untuk NLP sentiment).
3. **Cleaning diperlukan**: handling missing values, filter wilayah Kota Bandung, deduplikasi antar-anggota.
4. **Langkah selanjutnya**: Preprocessing di notebook `02_Preprocessing.ipynb`.